# Correlation Heatmap — 30 Model Variables

Pairwise Pearson correlations for the 30 numeric independent variables used in modelling.  
NaN values dropped **pairwise** so each correlation uses maximum available data.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR   = os.path.abspath(os.path.join('..', '..'))   # notebooks/eda/ -> repo root
DATA_PATH  = os.path.join(BASE_DIR, 'data', 'final', 'model_dataset.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'figures')
OUTPUT_PNG = os.path.join(OUTPUT_DIR, 'correlation_heatmap.png')
os.makedirs(OUTPUT_DIR, exist_ok=True)

variables = [
    # Episode rhythm features
    'episode_duration_sec', 'episode_beat_count', 'episode_rr_cv',
    # HR during episode duration
    'hr_dur_mean', 'hr_dur_std', 'hr_dur_min', 'hr_dur_slope',
    # SpO2 during episode duration
    'spo2_dur_mean', 'spo2_dur_std', 'spo2_dur_min', 'spo2_dur_slope',
    # MAP during episode duration
    'map_dur_mean', 'map_dur_std', 'map_dur_min', 'map_dur_slope',
    # Patient-level arrhythmia burden
    'total_episode_count', 'total_arrhythmia_burden_sec',
    'longest_episode_duration_sec', 'first_episode_start_sec',
    # Clinical
    'age', 'bmi', 'asa', 'preop_htn', 'preop_dm',
    'preop_hb', 'preop_plt', 'preop_na', 'preop_k',
    'preop_cr',
    'preop_alb'
]

In [2]:
df = pd.read_csv(DATA_PATH)
missing_vars = [v for v in variables if v not in df.columns]
if missing_vars:
    print(f'WARNING — not found, skipping: {missing_vars}')
use_cols = [v for v in variables if v in df.columns]
data = df[use_cols].copy()
print(f'Using {len(use_cols)} variables, {df.shape[0]} episodes')

Using 30 variables, 1284 episodes


In [3]:
corr = data.corr(method='pearson', min_periods=1)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    corr, ax=ax,
    annot=True, fmt='.2f', annot_kws={'size': 9},
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Pearson Correlation Coefficient', 'shrink': 0.75}
)
ax.set_title('Correlation Heatmap of Independent Variables', fontsize=14, pad=20)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

plt.tight_layout()
fig.savefig(OUTPUT_PNG, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {OUTPUT_PNG}')

Saved: c:\Users\sukka\Downloads\ioh-prediction\figures\correlation_heatmap.png


In [4]:
mask  = np.triu(np.ones(corr.shape, dtype=bool), k=1)
pairs = corr.where(mask).stack().reset_index()
pairs.columns = ['var_a', 'var_b', 'r']
pairs = pairs.sort_values('r', ascending=False).reset_index(drop=True)

print('TOP 5 POSITIVE:')
for _, row in pairs.head(5).iterrows():
    print(f'  {row["var_a"]:28s} x {row["var_b"]:28s}  r = {row["r"]:+.4f}')

print('\nTOP 5 NEGATIVE:')
for _, row in pairs.tail(5).iloc[::-1].iterrows():
    print(f'  {row["var_a"]:28s} x {row["var_b"]:28s}  r = {row["r"]:+.4f}')

TOP 5 POSITIVE:
  total_arrhythmia_burden_sec  x longest_episode_duration_sec  r = +0.8996
  episode_duration_sec         x longest_episode_duration_sec  r = +0.8967
  episode_duration_sec         x episode_beat_count            r = +0.8890
  map_dur_mean                 x map_dur_min                   r = +0.8821
  hr_dur_mean                  x hr_dur_min                    r = +0.8565

TOP 5 NEGATIVE:
  spo2_dur_std                 x spo2_dur_min                  r = -0.8505
  asa                          x preop_alb                     r = -0.4719
  spo2_dur_mean                x spo2_dur_std                  r = -0.4642
  episode_duration_sec         x total_episode_count           r = -0.4338
  asa                          x preop_hb                      r = -0.4048
